# Project Functionality Demo (Single Notebook Flow)

This notebook demonstrates the same backend processing in one linear file:
1. Ingest Excel sheets
2. Clean and prepare data
3. Transform to core variables
4. Run all BI analysis modules
5. Generate PDF and Excel reports
6. (Optional) Persist data to SQLite and reload

In [1]:
from pathlib import Path
import sys
from pprint import pprint

# Ensure project root is importable when running from notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from logger import setup_logging
setup_logging()

project_root

WindowsPath('C:/Users/Admin/Downloads/Missing Values and cleaning added/ecommerce_backend')

In [2]:
# Pick latest Excel file in data/raw (same bootstrap idea as main.py).
raw_dir = project_root / 'data' / 'raw'
excel_files = sorted(
    [p for p in list(raw_dir.glob('*.xlsx')) + list(raw_dir.glob('*.xls')) if not p.name.startswith('~$')],
    key=lambda p: p.stat().st_mtime,
 )

if not excel_files:
    raise FileNotFoundError(f'No valid Excel file found in: {raw_dir}')

file_path = excel_files[-1]
print(f'Using input file: {file_path}')

Using input file: C:\Users\Admin\Downloads\Missing Values and cleaning added\ecommerce_backend\data\raw\Data Structure.xlsx


## 1) Run The Same Pipeline Stages

In [3]:
# Run exact stage functions used by pipeline/runner.py
from pipeline.stage_01_ingest import ingest
from pipeline.stage_02_clean import clean
from pipeline.stage_03_transform import transform

context = {'file_path': str(file_path)}
context = ingest(context)
context = clean(context)
context = transform(context)

print('Pipeline stages complete.')
pprint(context['vars'])

2026-03-20 21:23:46 | INFO     | pipeline.stage_01_ingest | Ingesting: Data Structure.xlsx
2026-03-20 21:23:48 | INFO     | pipeline.stage_01_ingest | Loaded  Inventory:112  Sales:10000  Purchase:378  Expenses:12
2026-03-20 21:23:48 | INFO     | pipeline.stage_02_clean | Cleaned sales: removed 0 fully-empty rows
2026-03-20 21:23:48 | INFO     | pipeline.stage_02_clean | Cleaned purch: removed 0 fully-empty rows
2026-03-20 21:23:49 | INFO     | pipeline.stage_02_clean | Clean summary: inv -0 dup, sales -0 dup, purch -0 dup
2026-03-20 21:23:49 | INFO     | pipeline.stage_03_transform | Transform done. GrossRevenue=54,584,951  NetRevenue=53,019,062


Pipeline stages complete.
{'COGS': np.float64(43079360.11572498),
 'GrossRevenue': np.float64(54584951.0),
 'TotalDiscount': np.float64(1565889.0),
 'TotalOperatingExpense': np.float64(4366058.0),
 'TotalPurchaseCost': np.float64(25225144.0),
 'TotalQuantityBought': np.float64(11752.0),
 'TotalQuantitySold': np.float64(20070.0),
 'TotalRevenue': np.float64(53019062.0),
 'WeightedAvgCost': np.float64(2146.4554118447923)}


In [4]:
# Quick shape check after cleaning/transformation
print('inv   rows:', len(context['inv']))
print('sales rows:', len(context['sales']))
print('purch rows:', len(context['purch']))
print('exp   rows:', len(context['exp']))

inv   rows: 101
sales rows: 10000
purch rows: 378
exp   rows: 12


## 2) Build Store And Run All Analysis Functions

In [5]:
# Same store structure used by API routes/state.
store = {
    'inv': context['inv'],
    'sales': context['sales'],
    'purch': context['purch'],
    'exp': context['exp'],
    'vars': context['vars'],
    'ready': True,
}

In [6]:
from analysis.profitability import compute_profitability
from analysis.discounts import compute_discounts
from analysis.inventory import compute_inventory
from analysis.products import compute_products
from analysis.expenses import compute_expenses
from analysis.monthly_growth import compute_monthly_growth
from analysis.breakeven import compute_breakeven
from analysis.cashflow import compute_cashflow

results = {
    'profitability': compute_profitability(store),
    'discounts': compute_discounts(store),
    'inventory': compute_inventory(store),
    'products': compute_products(store),
    'expenses': compute_expenses(store),
    'monthly_growth': compute_monthly_growth(store),
    'breakeven': compute_breakeven(store),
    'cashflow': compute_cashflow(store),
}

print('Computed sections:', ', '.join(results.keys()))

2026-03-20 21:23:49 | INFO     | analysis.profitability | Profitability: NetProfit=5,573,644  GPM=18.75%
2026-03-20 21:23:49 | INFO     | analysis.discounts | Discounts: 2.87% of gross revenue
2026-03-20 21:23:49 | INFO     | analysis.inventory | Inventory: turnover=1.71x  DIO=213.7d
2026-03-20 21:23:49 | INFO     | analysis.products | Product metrics computed.
2026-03-20 21:23:49 | INFO     | analysis.expenses | Expenses: OpEx%=8.23
2026-03-20 21:23:49 | INFO     | analysis.monthly_growth | Monthly growth computed.
2026-03-20 21:23:49 | INFO     | analysis.breakeven | Break-even: BEU=1,586  MoS=92.10%
2026-03-20 21:23:49 | INFO     | analysis.cashflow | Cashflow: NetCashMovement=23,427,860


Computed sections: profitability, discounts, inventory, products, expenses, monthly_growth, breakeven, cashflow


In [7]:
# Print compact summaries
for section_name in [
    'profitability',
    'discounts',
    'inventory',
    'expenses',
    'breakeven',
    'cashflow',
]:
    print(f'\n=== {section_name.upper()} ===')
    pprint(results[section_name])


=== PROFITABILITY ===
{'cogs_npr': 43079360.12,
 'gross_profit_margin_pct': 18.75,
 'gross_profit_npr': 9939701.88,
 'gross_revenue_npr': 54584951.0,
 'net_profit_margin_pct': 10.51,
 'net_profit_npr': 5573643.88,
 'net_revenue_npr': 53019062.0,
 'total_discount_npr': 1565889.0,
 'total_opex_npr': 4366058.0}

=== DISCOUNTS ===
{'avg_discount_per_txn_npr': 702.51,
 'discount_pct_of_revenue': np.float64(2.87),
 'discounted_transactions': 2229,
 'discounted_txn_pct': 22.29,
 'monthly_discount': [{'MonthNum': 1, 'monthly_discount_npr': 64163.0},
                      {'MonthNum': 2, 'monthly_discount_npr': 69654.0},
                      {'MonthNum': 3, 'monthly_discount_npr': 102927.0},
                      {'MonthNum': 4, 'monthly_discount_npr': 132113.0},
                      {'MonthNum': 5, 'monthly_discount_npr': 52476.0},
                      {'MonthNum': 6, 'monthly_discount_npr': 93125.0},
                      {'MonthNum': 7, 'monthly_discount_npr': 65124.0},
                 

In [8]:
# Optional tabular views for product and monthly growth outputs
import pandas as pd

top_products_df = pd.DataFrame(results['products']['top_10_products_by_revenue'])
monthly_growth_df = pd.DataFrame(results['monthly_growth']['monthly'])

print('Top products:')
display(top_products_df.head(10))

print('Monthly growth:')
display(monthly_growth_df)

Top products:


,ItemID,ItemName,Category,ProductQtySold,ProductRevenue,ProductProfit,ContribPct
0,HOME006,Eureka Forbes Aquaguard Water Purifier,Home & Kitchen,119.0,2172358.0,1916929.81,4.10
1,ELEC011,HP 15s Intel Core i3 Laptop,Electronics,27.0,1890790.0,1832835.70,3.57
2,HOME011,Symphony Diet 12T Air Cooler,Home & Kitchen,103.0,1812044.0,1590959.09,3.42
3,HOME005,Philips Air Fryer HD9200,Home & Kitchen,124.0,1619585.0,1353424.53,3.05
4,HOME012,Livpure Smart Water Purifier,Home & Kitchen,88.0,1504068.0,1315179.92,2.84
5,FASH015,Arrow Formal Blazer Men,Grocery & Food,173.0,1348335.0,976998.21,2.54
6,FASH010,Adidas Ultraboost 22 Sneakers,Fashion & Clothing,123.0,1339409.0,1075394.98,2.53
7,ELEC002,Realme Narzo 70 Pro,Electronics,30.0,1218719.0,1154325.34,2.30
8,FASH009,Titan Raga Ladies Watch,Fashion & Clothing,139.0,1198095.0,899737.70,2.26
9,HLTH008,Omron HEM-7120 Digital BP Monitor,Health & Wellness,127.0,1143077.0,870477.16,2.16


Monthly growth:


,MonthName,MonthRevenue,MonthProfit,RevGrowth,ProfGrowth
0,Jan,3221421.0,278538.06,NaN,NaN
1,Feb,2670766.0,185902.50,-17.09,-33.26
2,Mar,3975849.0,432055.78,48.87,132.41
3,Apr,3839692.0,404510.87,-3.42,-6.38
4,May,4590620.0,526032.44,19.56,30.04
5,Jun,3379744.0,313975.53,-26.38,-40.31
6,Jul,3185912.0,264859.04,-5.74,-15.64
7,Aug,4116262.0,446412.59,29.20,68.55
8,Sep,5075625.0,582653.32,23.31,30.52
9,Oct,5616914.0,520541.00,10.66,-10.66


## 3) Generate Report Files

In [9]:
from reports.report_generator import generate_pdf, generate_excel

pdf_path = generate_pdf(store)
excel_path = generate_excel(store)

print('PDF report :', pdf_path)
print('Excel report:', excel_path)

ModuleNotFoundError: No module named 'fpdf'

## 4) Optional SQLite Persistence Check

In [ ]:
from database import init_db, merge_context, load_store

init_db()
merge_context(context, source_filename=file_path.name)
reloaded_store = load_store()

print('Reloaded ready status:', reloaded_store['ready'])
print('Reloaded vars available:', reloaded_store['vars'] is not None)

2026-03-20 21:04:03 | INFO     | pipeline.stage_03_transform | Transform done. GrossRevenue=54,584,951  NetRevenue=53,019,062


Reloaded ready status: True
Reloaded vars available: True
